# Rope Flow — Full Pipeline V10.4 (Fully Supervised)

**Course:** MECH 798M / EECE 798K — Data-Driven Modeling  
**Pipeline:**
- Supervised cycle-level classification using time-aligned JSON labels
- Fixed-window cycle extraction centered on paired `||omega||` peaks
- Input shape per sample: `12 x WINDOW` (two devices, six channels each)
- LOSO evaluation for `RF` and `PCA+GBM` only
- No clustering or pseudo-labeling

In [12]:
import os, glob, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from bisect import bisect_right
from scipy.signal import savgol_filter, find_peaks

from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
 )
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# -- Paths ----------------------------------------------------------------
DATA_PROCESSED = os.path.join('..', '..', 'data', 'processed')
LABELED_RAW_BASE = os.path.join('..', '..', 'data', 'raw', 'new-labeled-sessions')

# -- Config ---------------------------------------------------------------
DIRECT_CFG = {
    'FS':                 100.0,
    'WINDOW':             64,
    'PEAK_PROM_DEGS':     50.0,
    'PEAK_MIN_DEGS':      50.0,
    'PEAK_SAVGOL_WINDOW': 15,
    'PEAK_SAVGOL_POLY':   3,
    'PEAK_MIN_PERIOD_S':  0.2,
    'MERGE_GAP_S':        0.15,
    'PCA_VAR':            0.99,
    # Put your exact 12 supervised classes here; all other labels map to UNKNOWN_CLASS.
    'SUPERVISED_CLASSES': [
        'dragon_roll',
        'underhand_right',
        'underhand_left',
        'overhand_left',
        'overhand_right',
        'sneak_underhand_left',
        'sneak_underhand_right',
        'sneak_overhand_left',
        'sneak_overhand_right',
        'clockwise',
        'counter_clockwise'
    ],
    'UNKNOWN_CLASS':      'unknown',
}
if len(DIRECT_CFG['SUPERVISED_CLASSES']) != 11:
    raise ValueError('DIRECT_CFG[SUPERVISED_CLASSES] must contain exactly 11 known classes.')

# -- Results directory (new folder per run) --------------------------------
import datetime
RUN_NAME    = datetime.datetime.now().strftime('run_%Y%m%d_%H%M%S')
RESULTS_DIR = os.path.join('..', '..', 'results', 'Full_pipeline', RUN_NAME)
os.makedirs(RESULTS_DIR, exist_ok=True)
print(f'Results will be saved to: {RESULTS_DIR}')

Results will be saved to: ..\..\results\Full_pipeline\run_20260426_152714


### 1. Session discovery and cycle detection

In [13]:
# -- Session discovery ---------------------------------------------------
def discover_direct_sessions(processed_dir):
    entries = []
    for d0 in sorted(glob.glob(os.path.join(processed_dir, '*_device0_processed.csv'))):
        d1 = d0.replace('_device0_', '_device1_')
        if not os.path.isfile(d1):
            continue
        stem = os.path.basename(d0).replace('_device0_processed.csv', '')
        entries.append((d0, d1, stem))
    return entries

# -- Label canonicalization ----------------------------------------------
LABEL_ALIAS_MAP = {
    'bf': 'dragon_roll', 'bf2': 'dragon_roll', 'fb': 'dragon_roll', 'fb2': 'dragon_roll',
    'dragon roll': 'dragon_roll', 'dragon_roll': 'dragon_roll',
    'ur': 'underhand_right', 'ur0': 'underhand_right',
    'underhand right': 'underhand_right', 'underhand': 'underhand_right',
    'ul': 'underhand_left', 'ul0': 'underhand_left',
    'underhand left': 'underhand_left',
    'ol': 'overhand_left', 'ol0': 'overhand_left', 'ol2': 'overhand_left',
    'overhand left': 'overhand_left', 'overhand': 'overhand_left',
    'or': 'overhand_right', 'or2': 'overhand_right', 'or3': 'overhand_right',
    'overhand right': 'overhand_right',
    'usl': 'sneak_underhand_left', 'sneak underhand left': 'sneak_underhand_left',
    'usr': 'sneak_underhand_right', 'sneak underhand right': 'sneak_underhand_right',
    'osl': 'sneak_overhand_left',  'sneak overhand left': 'sneak_overhand_left',
    'osr': 'sneak_overhand_right', 'sneak overhand right': 'sneak_overhand_right',
    'sneak overhand': 'sneak_overhand', 'sneak underhand': 'sneak_underhand',
    'cw': 'clockwise', 'clockwise': 'clockwise',
    'ccw': 'counter_clockwise', 'counter clockwise': 'counter_clockwise',
    'idle': 'idle', 'idle3': 'idle', 'no movement': 'idle',
    'vq5': 'vq', 'vq15': 'vq', 'vq16': 'vq', 'vq': 'vq',
    'excluded': 'excluded',
}

def _normalize_label_key(label):
    s = str(label).strip().lower().replace('_', ' ').replace('-', ' ')
    return ' '.join(s.split())

def canonicalize_label(label):
    key = _normalize_label_key(label)
    if key in LABEL_ALIAS_MAP:
        return LABEL_ALIAS_MAP[key]
    for sep in ('/', '|'):
        if sep in key:
            parts = [p.strip() for p in key.split(sep) if p.strip()]
            if parts:
                first = _normalize_label_key(parts[0])
                return LABEL_ALIAS_MAP.get(first, first)
    return key

def _safe_float(x):
    if x is None:
        return None
    try:
        return float(x)
    except (TypeError, ValueError):
        return None

# -- Cycle-extraction helpers ---------------------------------------------
def _smooth_mag_deg(omega_rad, cfg):
    mag_deg = np.linalg.norm(omega_rad, axis=1) * (180.0 / np.pi)
    n = len(mag_deg)
    if n < 7:
        return mag_deg
    win = int(cfg['PEAK_SAVGOL_WINDOW'])
    if win % 2 == 0:
        win += 1
    max_odd = n if n % 2 == 1 else n - 1
    win = max(5, min(win, max_odd))
    poly = max(1, min(int(cfg['PEAK_SAVGOL_POLY']), win - 2))
    return savgol_filter(mag_deg, window_length=win, polyorder=poly, mode='interp')

def detect_cycle_peaks(omega_rad, fs, cfg):
    mag_smooth = _smooth_mag_deg(omega_rad, cfg)
    peaks, _ = find_peaks(
        mag_smooth,
        distance=max(1, int(cfg['PEAK_MIN_PERIOD_S'] * fs)),
        prominence=cfg['PEAK_PROM_DEGS'],
    )
    peaks = np.array([int(p) for p in peaks if mag_smooth[p] >= cfg['PEAK_MIN_DEGS']], dtype=int)
    return peaks, mag_smooth

def merge_device_peaks_pairs(peaks0, peaks1, fs_or_gap=100.0, gap_s=None):
    if gap_s is None:
        fs = float(DIRECT_CFG.get('FS', 100.0))
        gap_s = float(fs_or_gap)
    else:
        fs = float(fs_or_gap)
        gap_s = float(gap_s)

    tagged = [(p / fs, p, 'D0') for p in peaks0] + [(p / fs, p, 'D1') for p in peaks1]
    if not tagged:
        return []
    tagged.sort(key=lambda x: x[0])
    all_idx_t = [x[0] for x in tagged]

    accepted = [0]
    for i in range(1, len(all_idx_t)):
        if all_idx_t[i] - all_idx_t[accepted[-1]] > gap_s:
            accepted.append(i)

    group_peaks = [{} for _ in accepted]
    a_idx = 0
    for i, (_, peak_idx, src) in enumerate(tagged):
        if a_idx + 1 < len(accepted) and i >= accepted[a_idx + 1]:
            a_idx += 1
        group_peaks[a_idx].setdefault(src, peak_idx)

    return [(g['D0'], g['D1']) for g in group_peaks if 'D0' in g and 'D1' in g]

def extract_fixed_window(ch6, center_idx, window=64):
    half = int(window // 2)
    start = int(center_idx) - half
    end = start + int(window)
    out = np.zeros((6, int(window)), dtype=np.float32)
    src_lo = max(0, start)
    src_hi = min(ch6.shape[0], end)
    if src_hi <= src_lo:
        return out
    dst_lo = src_lo - start
    out[:, dst_lo:dst_lo + (src_hi - src_lo)] = ch6[src_lo:src_hi].T
    return out

def load_session(path_d0, path_d1):
    d0 = pd.read_csv(path_d0)
    d1 = pd.read_csv(path_d1)
    return d0, d1

def extract_signals(df):
    t = df['timestamp_ms'].values / 1000.0
    A = df[['ax_w', 'ay_w', 'az_w']].values
    omega = df[['gx', 'gy', 'gz']].values * (np.pi / 180.0)
    return t, A, omega

# -- Annotation loading & time labeling -----------------------------------
def _load_time_labels_for_session(session_id):
    session_dir = os.path.join(LABELED_RAW_BASE, session_id)
    if not os.path.isdir(session_dir):
        return None
    for name in ('labels_corrected.json', 'labels.json', 'labels_vad.json'):
        path = os.path.join(session_dir, name)
        if not os.path.isfile(path):
            continue
        with open(path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        segments = data.get('segments', []) or []
        label_events = data.get('label_events', []) or []

        clean_segs = []
        for s in segments:
            if not isinstance(s, dict):
                continue
            if not {'start', 'end', 'label'}.issubset(s):
                continue
            t0 = _safe_float(s.get('start'))
            t1 = _safe_float(s.get('end'))
            if t0 is None or t1 is None:
                continue
            if not np.isfinite(t0) or not np.isfinite(t1):
                continue
            if t1 <= t0:
                continue
            clean_segs.append((t0, t1, canonicalize_label(s.get('label'))))

        clean_evs = []
        for ev in label_events:
            if not isinstance(ev, dict):
                continue
            if 'time' not in ev or 'label' not in ev:
                continue
            te = _safe_float(ev.get('time'))
            if te is None or (not np.isfinite(te)):
                continue
            clean_evs.append((te, canonicalize_label(ev.get('label'))))
        clean_evs.sort(key=lambda x: x[0])

        if not clean_segs and not clean_evs:
            continue

        return {
            'json_path': path,
            'source_name': name,
            'segments': clean_segs,
            'events': clean_evs,
        }
    return None

def _label_at_time(t_s, ann):
    for t0, t1, lab in ann['segments']:
        if t0 <= t_s < t1:
            return lab
    if ann['segments']:
        centers = np.array([(a + b) * 0.5 for a, b, _ in ann['segments']])
        return ann['segments'][int(np.argmin(np.abs(centers - t_s)))][2]
    if ann['events']:
        times = [e[0] for e in ann['events']]
        return ann['events'][max(0, bisect_right(times, t_s) - 1)][1]
    return None

def map_to_supervised_class(raw_label, cfg):
    if raw_label is None:
        return None
    lab = canonicalize_label(raw_label)
    if lab == 'excluded':
        return None
    known = set(cfg['SUPERVISED_CLASSES'])
    if lab in known:
        return lab
    return cfg['UNKNOWN_CLASS']

def extract_labeled_cycles_from_entry(entry, cfg, ann):
    d0_path, d1_path, _ = entry
    d0, d1 = load_session(d0_path, d1_path)
    t0, A0, om0 = extract_signals(d0)
    t1, A1, om1 = extract_signals(d1)
    peaks0, _ = detect_cycle_peaks(om0, cfg['FS'], cfg)
    peaks1, _ = detect_cycle_peaks(om1, cfg['FS'], cfg)
    pairs = merge_device_peaks_pairs(peaks0, peaks1, cfg['FS'], cfg.get('MERGE_GAP_S', 0.15))

    ch0 = np.column_stack([A0, om0 * (180.0 / np.pi)])
    ch1 = np.column_stack([A1, om1 * (180.0 / np.pi)])

    mats, labels, mids = [], [], []
    for p0, p1 in pairs:
        t_mid = 0.5 * (t0[int(p0)] + t1[int(p1)])
        y_raw = _label_at_time(float(t_mid), ann)
        y_lab = map_to_supervised_class(y_raw, cfg)
        if y_lab is None:
            continue

        w0 = extract_fixed_window(ch0, int(p0), cfg['WINDOW'])
        w1 = extract_fixed_window(ch1, int(p1), cfg['WINDOW'])
        mats.append(np.vstack([w0, w1]).astype(np.float32))
        labels.append(y_lab)
        mids.append(float(t_mid))
    return mats, labels, mids

def build_labeled_cycle_dataset(entries, cfg):
    X_list, y_list, g_list, sid_list, tm_list = [], [], [], [], []
    ann_cache = {}
    n_sessions_with_labels = 0

    for entry in entries:
        sid = entry[2]
        if sid not in ann_cache:
            ann_cache[sid] = _load_time_labels_for_session(sid)
        ann = ann_cache[sid]
        if ann is None:
            continue
        mats, labs, mids = extract_labeled_cycles_from_entry(entry, cfg, ann)
        if not mats:
            continue
        n_sessions_with_labels += 1
        for m, y, tm in zip(mats, labs, mids):
            X_list.append(m)
            y_list.append(y)
            g_list.append(sid)
            sid_list.append(sid)
            tm_list.append(tm)

    if not X_list:
        raise RuntimeError('No labeled cycles extracted.')

    X = np.stack(X_list).astype(np.float32)
    y = np.array(y_list, dtype=object)
    groups = np.array(g_list, dtype=object)
    session_ids = np.array(sid_list, dtype=object)
    t_mid_s = np.array(tm_list, dtype=np.float32)

    print(f'Sessions found: {len(entries)}')
    print(f'Sessions with labels and cycles: {n_sessions_with_labels}')
    print(f'Labeled dataset: X={X.shape} | classes={len(np.unique(y))}')
    return X, y, groups, session_ids, t_mid_s

### 2. Feature building and supervised LOSO evaluation

In [14]:
def build_feature_matrix(X_cycles):
    # Pure time-series representation: flatten 12 x WINDOW into 1D vector per cycle.
    F = X_cycles.reshape(len(X_cycles), -1).astype(np.float32)
    print(f'Feature matrix: {F.shape} (12 x WINDOW flattened)')
    return F

def build_class_list(cfg):
    classes = list(cfg['SUPERVISED_CLASSES'])
    unk = str(cfg.get('UNKNOWN_CLASS', 'unknown'))
    if unk not in classes:
        classes.append(unk)
    return classes

def encode_labels(y_str, class_names):
    cls_to_idx = {c: i for i, c in enumerate(class_names)}
    y_idx = np.array([cls_to_idx.get(y, cls_to_idx[class_names[-1]]) for y in y_str], dtype=np.int32)
    return y_idx, cls_to_idx

def _save_supervised_eval(y_true, y_pred, class_names, tag, results_dir):
    acc = accuracy_score(y_true, y_pred)
    macro_f1 = f1_score(y_true, y_pred, average='macro', zero_division=0)

    report = classification_report(
        y_true,
        y_pred,
        labels=np.arange(len(class_names)),
        target_names=class_names,
        zero_division=0,
        output_dict=True,
    )

    per_class_rows = []
    for cname in class_names:
        stats = report.get(cname, {'precision': 0.0, 'recall': 0.0, 'f1-score': 0.0, 'support': 0.0})
        per_class_rows.append({
            'class': cname,
            'precision': float(stats['precision']),
            'recall': float(stats['recall']),
            'f1': float(stats['f1-score']),
            'support': int(stats['support']),
        })

    df_per_class = pd.DataFrame(per_class_rows)
    out_pc = os.path.join(results_dir, f'{tag}_per_class_metrics.csv')
    df_per_class.to_csv(out_pc, index=False)

    cm = confusion_matrix(y_true, y_pred, labels=np.arange(len(class_names)))
    fig, ax = plt.subplots(figsize=(10, 8))
    ConfusionMatrixDisplay(cm, display_labels=class_names).plot(
        ax=ax, xticks_rotation=45, cmap='Blues', colorbar=False
    )
    ax.set_title(f'{tag} | acc={acc:.3f}, macro-F1={macro_f1:.3f}')
    fig.tight_layout()
    out_cm = os.path.join(results_dir, f'fig_confusion_{tag}.png')
    fig.savefig(out_cm, bbox_inches='tight')
    plt.close(fig)

    print(f'[{tag}] accuracy={acc:.4f} | macro-F1={macro_f1:.4f}')
    print(df_per_class.to_string(index=False))

    return {
        'approach': tag,
        'accuracy': float(acc),
        'macro_f1': float(macro_f1),
        'per_class_csv': out_pc,
        'confusion_png': out_cm,
    }

def run_loso_pca_gbm(X_feat, y, groups, class_names, cfg, results_dir):
    uniq = np.unique(groups)
    y_true_all, y_pred_all = [], []

    for fi, g in enumerate(uniq, 1):
        tr, te = groups != g, groups == g
        Xtr, Xte = X_feat[tr], X_feat[te]
        ytr, yte = y[tr], y[te]
        if len(yte) == 0:
            continue

        if len(np.unique(ytr)) < 2:
            pred = np.full_like(yte, fill_value=int(ytr[0]))
            y_true_all.extend(yte.tolist())
            y_pred_all.extend(pred.tolist())
            print(f'  [PCA+GBM] fold {fi}/{len(uniq)} (single-class fallback)')
            continue

        sc = StandardScaler()
        Xts = sc.fit_transform(Xtr)
        Xes = sc.transform(Xte)

        pca = PCA(n_components=float(cfg.get('PCA_VAR', 0.99)), svd_solver='full')
        Xtp = pca.fit_transform(Xts)
        Xep = pca.transform(Xes)

        clf = HistGradientBoostingClassifier(
            max_iter=200,
            max_depth=6,
            learning_rate=0.08,
            l2_regularization=1.0,
            random_state=42,
        )
        clf.fit(Xtp, ytr)
        pred = clf.predict(Xep)

        y_true_all.extend(yte.tolist())
        y_pred_all.extend(pred.tolist())
        print(f'  [PCA+GBM] fold {fi}/{len(uniq)}')

    y_true = np.array(y_true_all, dtype=np.int32)
    y_pred = np.array(y_pred_all, dtype=np.int32)
    return _save_supervised_eval(y_true, y_pred, class_names, 'pca_gbm_loso', results_dir)

def run_loso_rf(X_feat, y, groups, class_names, results_dir):
    uniq = np.unique(groups)
    y_true_all, y_pred_all = [], []

    for fi, g in enumerate(uniq, 1):
        tr, te = groups != g, groups == g
        Xtr, Xte = X_feat[tr], X_feat[te]
        ytr, yte = y[tr], y[te]
        if len(yte) == 0:
            continue

        if len(np.unique(ytr)) < 2:
            pred = np.full_like(yte, fill_value=int(ytr[0]))
            y_true_all.extend(yte.tolist())
            y_pred_all.extend(pred.tolist())
            print(f'  [RF] fold {fi}/{len(uniq)} (single-class fallback)')
            continue

        sc = StandardScaler()
        Xts = sc.fit_transform(Xtr)
        Xes = sc.transform(Xte)

        clf = RandomForestClassifier(
            n_estimators=400,
            max_depth=24,
            min_samples_leaf=1,
            class_weight='balanced_subsample',
            n_jobs=-1,
            random_state=42,
        )
        clf.fit(Xts, ytr)
        pred = clf.predict(Xes)

        y_true_all.extend(yte.tolist())
        y_pred_all.extend(pred.tolist())
        print(f'  [RF] fold {fi}/{len(uniq)}')

    y_true = np.array(y_true_all, dtype=np.int32)
    y_pred = np.array(y_pred_all, dtype=np.int32)
    return _save_supervised_eval(y_true, y_pred, class_names, 'rf_loso', results_dir)

### 3. Dataset diagnostics

In [15]:
def save_dataset_diagnostics(y_str, groups, results_dir):
    df = pd.DataFrame({'label': y_str, 'session_id': groups})

    label_counts = df['label'].value_counts().rename_axis('label').reset_index(name='count')
    label_counts.to_csv(os.path.join(results_dir, 'label_distribution.csv'), index=False)

    session_counts = df['session_id'].value_counts().rename_axis('session_id').reset_index(name='count')
    session_counts.to_csv(os.path.join(results_dir, 'session_cycle_counts.csv'), index=False)

    fig, ax = plt.subplots(figsize=(10, 4))
    ax.bar(label_counts['label'], label_counts['count'], color='#2f6f8f')
    ax.set_title('Cycle counts per class')
    ax.set_xlabel('Class')
    ax.set_ylabel('Count')
    ax.tick_params(axis='x', rotation=45)
    fig.tight_layout()
    fig.savefig(os.path.join(results_dir, 'fig_label_distribution.png'), bbox_inches='tight')
    plt.close(fig)

    print('Saved dataset diagnostics:')
    print(f"  {os.path.join(results_dir, 'label_distribution.csv')}")
    print(f"  {os.path.join(results_dir, 'session_cycle_counts.csv')}")
    print(f"  {os.path.join(results_dir, 'fig_label_distribution.png')}")

### 4. Main: build supervised dataset

In [16]:
# ════════════════════════════════════════════════════════════════════════════
# MAIN (SUPERVISED DATASET BUILD)
# ════════════════════════════════════════════════════════════════════════════
SESSIONS = discover_direct_sessions(DATA_PROCESSED)
print(f'Sessions found: {len(SESSIONS)}')

X_cycles, y_labels, groups, session_ids, t_mid_s = build_labeled_cycle_dataset(SESSIONS, DIRECT_CFG)
X_feat = build_feature_matrix(X_cycles)

CLASS_NAMES = build_class_list(DIRECT_CFG)
y_idx, CLASS_TO_IDX = encode_labels(y_labels, CLASS_NAMES)

print(f'Using {len(CLASS_NAMES)} classes (including unknown):')
print(CLASS_NAMES)

save_dataset_diagnostics(y_labels, groups, RESULTS_DIR)

supervised_dataset = pd.DataFrame({
    'session_id': session_ids,
    't_mid_s': t_mid_s,
    'label': y_labels,
    'label_idx': y_idx,
})
out_ds = os.path.join(RESULTS_DIR, 'supervised_cycle_index.csv')
supervised_dataset.to_csv(out_ds, index=False)
print(f'Supervised cycle index saved: {out_ds}')

Sessions found: 51
Sessions found: 51
Sessions with labels and cycles: 15
Labeled dataset: X=(1968, 12, 64) | classes=12
Feature matrix: (1968, 768) (12 x WINDOW flattened)
Using 12 classes (including unknown):
['dragon_roll', 'underhand_right', 'underhand_left', 'overhand_left', 'overhand_right', 'sneak_underhand_left', 'sneak_underhand_right', 'sneak_overhand_left', 'sneak_overhand_right', 'clockwise', 'counter_clockwise', 'unknown']
Saved dataset diagnostics:
  ..\..\results\Full_pipeline\run_20260426_152714\label_distribution.csv
  ..\..\results\Full_pipeline\run_20260426_152714\session_cycle_counts.csv
  ..\..\results\Full_pipeline\run_20260426_152714\fig_label_distribution.png
Supervised cycle index saved: ..\..\results\Full_pipeline\run_20260426_152714\supervised_cycle_index.csv


### 5. Main: supervised LOSO (RF and PCA+GBM)

In [9]:
print('\n-- LOSO: PCA + GBM --')
res_pca = run_loso_pca_gbm(X_feat, y_idx, groups, CLASS_NAMES, DIRECT_CFG, RESULTS_DIR)

print('\n-- LOSO: Random Forest --')
res_rf = run_loso_rf(X_feat, y_idx, groups, CLASS_NAMES, RESULTS_DIR)

summary = pd.DataFrame([
    {'approach': 'PCA + GBM', 'accuracy': res_pca['accuracy'], 'macro_f1': res_pca['macro_f1']},
    {'approach': 'Random Forest', 'accuracy': res_rf['accuracy'], 'macro_f1': res_rf['macro_f1']},
]).sort_values('macro_f1', ascending=False).reset_index(drop=True)
summary.index += 1

print('\n' + '=' * 56)
print('LOSO supervised performance (cycle-level)')
print('=' * 56)
print(summary.to_string())
print('=' * 56)

out_summary = os.path.join(RESULTS_DIR, 'loso_supervised_summary.csv')
summary.to_csv(out_summary, index=False)
print(f'Summary saved: {out_summary}')

fig, ax = plt.subplots(figsize=(7, 3))
ax.barh(summary['approach'][::-1], summary['macro_f1'][::-1], color=['#4c956c', '#2c6e91'])
ax.set_xlabel('Macro-F1')
ax.set_xlim(0, 1.0)
ax.set_title('LOSO macro-F1 comparison (supervised)')
for i, row in summary[::-1].reset_index(drop=True).iterrows():
    ax.text(row['macro_f1'] + 0.01, i, f"{row['macro_f1']:.3f}", va='center', fontsize=9)
fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, 'fig_loso_supervised_comparison.png'), bbox_inches='tight')
plt.close(fig)

print('Results saved to', RESULTS_DIR)


-- LOSO: PCA + GBM --
  [PCA+GBM] fold 1/15
  [PCA+GBM] fold 2/15
  [PCA+GBM] fold 3/15
  [PCA+GBM] fold 4/15
  [PCA+GBM] fold 5/15
  [PCA+GBM] fold 6/15
  [PCA+GBM] fold 7/15
  [PCA+GBM] fold 8/15
  [PCA+GBM] fold 9/15
  [PCA+GBM] fold 10/15
  [PCA+GBM] fold 11/15
  [PCA+GBM] fold 12/15
  [PCA+GBM] fold 13/15
  [PCA+GBM] fold 14/15
  [PCA+GBM] fold 15/15
[pca_gbm_loso] accuracy=0.4421 | macro-F1=0.2935
                class  precision   recall       f1  support
          dragon_roll   0.455172 0.298643 0.360656      221
      underhand_right   0.357942 0.434783 0.392638      368
       underhand_left   0.000000 0.000000 0.000000       18
        overhand_left   0.471562 0.756219 0.580892      603
       overhand_right   0.000000 0.000000 0.000000       57
 sneak_underhand_left   1.000000 0.230769 0.375000       26
sneak_underhand_right   0.666667 0.357143 0.465116       56
  sneak_overhand_left   0.607595 0.494845 0.545455       97
 sneak_overhand_right   1.000000 0.117647 0.210526  

In [17]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.linear_model import RidgeClassifierCV
from sktime.transformations.panel.rocket import MiniRocket

# ── PyTorch Wrapper for seamless Sklearn-like integration ─────────────
class TorchWrapper:
    def __init__(self, model, epochs=50, batch_size=32, lr=1e-3):
        self.model = model
        self.epochs = epochs
        self.batch_size = batch_size
        self.criterion = nn.CrossEntropyLoss()
        self.optimizer = optim.Adam(self.model.parameters(), lr=lr)
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model.to(self.device)

    def fit(self, X, y):
        # Expects X: (N, C, T)
        dataset = TensorDataset(torch.tensor(X, dtype=torch.float32), torch.tensor(y, dtype=torch.long))
        loader = DataLoader(dataset, batch_size=self.batch_size, shuffle=True)
        
        self.model.train()
        for _ in range(self.epochs):
            for batch_X, batch_y in loader:
                batch_X, batch_y = batch_X.to(self.device), batch_y.to(self.device)
                self.optimizer.zero_grad()
                outputs = self.model(batch_X)
                loss = self.criterion(outputs, batch_y)
                loss.backward()
                self.optimizer.step()
        return self

    def predict(self, X):
        self.model.eval()
        with torch.no_grad():
            X_tensor = torch.tensor(X, dtype=torch.float32).to(self.device)
            outputs = self.model(X_tensor)
            return torch.argmax(outputs, dim=1).cpu().numpy()

# ── Architectures ──────────────────────────────────────────────────────
class CNN1D(nn.Module):
    def __init__(self, in_channels, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_channels, 64, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.MaxPool1d(2),
            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(128, num_classes)
        )
    def forward(self, x): return self.net(x)

class LSTMModel(nn.Module):
    def __init__(self, in_channels, num_classes, hidden_size=64):
        super().__init__()
        self.lstm = nn.LSTM(in_channels, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, num_classes)
    
    def forward(self, x):
        # PyTorch LSTM expects (N, T, C), so we permute from (N, C, T)
        x = x.permute(0, 2, 1) 
        _, (h_n, _) = self.lstm(x)
        return self.fc(h_n[-1])

# ── Main LOSO Pipeline ─────────────────────────────────────────────────
def run_loso_classifiers(X_feat, y, groups, class_names, cfg, results_dir, model_type='minirocket'):
    """
    X_feat shape: (N_samples, C_channels, T_timesteps)
    model_type options: 'minirocket', 'svm', '1d_cnn', 'lstm'
    """
    uniq = np.unique(groups)
    y_true_all, y_pred_all = [], []
    num_classes = len(np.unique(y))
    in_channels = X_feat.shape[1]

    for fi, g in enumerate(uniq, 1):
        tr, te = groups != g, groups == g
        Xtr, Xte = X_feat[tr], X_feat[te]
        ytr, yte = y[tr], y[te]
        
        if len(yte) == 0: continue

        if len(np.unique(ytr)) < 2:
            pred = np.full_like(yte, fill_value=int(ytr[0]))
            y_true_all.extend(yte.tolist())
            y_pred_all.extend(pred.tolist())
            print(f'  [{model_type.upper()}] fold {fi}/{len(uniq)} (single-class fallback)')
            continue

        # Feature Scaling (Channel-wise across time)
        N_tr, C, T = Xtr.shape
        N_te = Xte.shape[0]
        
        sc = StandardScaler()
        # Flatten temporal dimension to scale properly, then reshape back
        Xtr_scaled = sc.fit_transform(Xtr.transpose(0, 2, 1).reshape(-1, C)).reshape(N_tr, T, C).transpose(0, 2, 1)
        Xte_scaled = sc.transform(Xte.transpose(0, 2, 1).reshape(-1, C)).reshape(N_te, T, C).transpose(0, 2, 1)

        # Model Initialization & Training
        if model_type == 'minirocket':
            # MiniRocket expects (N, C, T)
            transformer = MiniRocket()
            Xtr_trans = transformer.fit_transform(Xtr_scaled)
            Xte_trans = transformer.transform(Xte_scaled)
            clf = RidgeClassifierCV(alphas=np.logspace(-3, 3, 10))
            clf.fit(Xtr_trans, ytr)
            pred = clf.predict(Xte_trans)

        elif model_type == 'svm':
            # SVM expects flattened 2D arrays
            clf = SVC(kernel='rbf', class_weight='balanced')
            clf.fit(Xtr_scaled.reshape(N_tr, -1), ytr)
            pred = clf.predict(Xte_scaled.reshape(N_te, -1))

        elif model_type == '1d_cnn':
            clf = TorchWrapper(CNN1D(in_channels, num_classes))
            clf.fit(Xtr_scaled, ytr)
            pred = clf.predict(Xte_scaled)

        elif model_type == 'lstm':
            clf = TorchWrapper(LSTMModel(in_channels, num_classes))
            clf.fit(Xtr_scaled, ytr)
            pred = clf.predict(Xte_scaled)

        else:
            raise ValueError(f"Unknown model_type: {model_type}")

        y_true_all.extend(yte.tolist())
        y_pred_all.extend(pred.tolist())
        print(f'  [{model_type.upper()}] fold {fi}/{len(uniq)}')

    y_true = np.array(y_true_all, dtype=np.int32)
    y_pred = np.array(y_pred_all, dtype=np.int32)
    
    return _save_supervised_eval(y_true, y_pred, class_names, f'{model_type}_loso', results_dir)

In [18]:
# ── Multi-Model LOSO Evaluation ────────────────────────────────────────
print('\n-- LOSO: Deep Learning & Time-Series Classifiers --')

# Ensure X_input is (N, Channels, Time) for the newer models
# If X_cycles is (N, 60, 12), transpose it to (N, 12, 60)
X_input = X_cycles.transpose(0, 2, 1) if X_cycles.shape[1] > X_cycles.shape[2] else X_cycles

models_to_run = ['svm', '1d_cnn', 'lstm']

for model_name in models_to_run:
    print(f"\n[Evaluating {model_name.upper()}]...")
    try:
        run_loso_classifiers(
            X_feat=X_input, 
            y=y_idx, 
            groups=groups, 
            class_names=CLASS_NAMES, 
            cfg=DIRECT_CFG, 
            results_dir=RESULTS_DIR,
            model_type=model_name
        )
    except Exception as e:
        print(f"  [Error] {model_name} failed: {e}")


-- LOSO: Deep Learning & Time-Series Classifiers --

[Evaluating SVM]...
  [SVM] fold 1/15
  [SVM] fold 2/15
  [SVM] fold 3/15
  [SVM] fold 4/15
  [SVM] fold 5/15
  [SVM] fold 6/15
  [SVM] fold 7/15
  [SVM] fold 8/15
  [SVM] fold 9/15
  [SVM] fold 10/15
  [SVM] fold 11/15
  [SVM] fold 12/15
  [SVM] fold 13/15
  [SVM] fold 14/15
  [SVM] fold 15/15
[svm_loso] accuracy=0.4553 | macro-F1=0.4087
                class  precision   recall       f1  support
          dragon_roll   0.365854 0.610860 0.457627      221
      underhand_right   0.433225 0.361413 0.394074      368
       underhand_left   0.000000 0.000000 0.000000       18
        overhand_left   0.587332 0.507463 0.544484      603
       overhand_right   0.156250 0.263158 0.196078       57
 sneak_underhand_left   0.647059 0.423077 0.511628       26
sneak_underhand_right   0.756098 0.553571 0.639175       56
  sneak_overhand_left   0.571429 0.536082 0.553191       97
 sneak_overhand_right   0.833333 0.294118 0.434783       17
     